# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
import torch
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])
def predict(w1, w2, w3):
    return w1 * x + w2 * x**2 + w3
def calculate_loss(y_pred):
    return torch.abs(y - y_pred).mean()
w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
w3 = torch.tensor(1.0, requires_grad=True)

y_pred = predict(w1, w2, w3)
loss = calculate_loss(y_pred)

print("当前 w1：", w1.item())
print("当前 w2：", w2.item())
print("当前 w3：", w3.item())
print("预测值：", y_pred.detach().tolist())
print("损失：", loss.item())

w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
w3 = torch.tensor(1.0, requires_grad=True)

optimizer = torch.optim.SGD([w1, w2, w3], lr=0.1)

optimizer.zero_grad()

y_pred = predict(w1, w2, w3)
loss = calculate_loss(y_pred)
loss.backward()

print("\n--- 更新前 ---")
print("w1：", w1.item())
print("w2：", w2.item())
print("w3：", w3.item())
print("损失：", loss.item())
print("w1 的梯度：", w1.grad.item())
print("w2 的梯度：", w2.grad.item())
print("w3 的梯度：", w3.grad.item())

optimizer.step()

with torch.no_grad():
    new_pred = predict(w1, w2, w3)
    new_loss = calculate_loss(new_pred)

print("\n--- 更新后 ---")
print("w1：", w1.item())
print("w2：", w2.item())
print("w3：", w3.item())
print("预测值：", new_pred.tolist())
print("损失：", new_loss.item())

def train_w(initial_w=1.0, epochs=1000, base_lr=0.1, decay=True):
    w1 = torch.tensor(float(initial_w), requires_grad=True)
    w2 = torch.tensor(float(initial_w), requires_grad=True)
    w3 = torch.tensor(float(initial_w), requires_grad=True)

    optimizer = torch.optim.SGD([w1, w2, w3], lr=base_lr)
    history = []

    for epoch in range(1, epochs + 1):
        lr = base_lr / epoch if decay else base_lr
        optimizer.param_groups[0]["lr"] = lr

        optimizer.zero_grad()

        y_pred = predict(w1, w2, w3)
        loss = calculate_loss(y_pred)

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            current_loss = calculate_loss(
                predict(w1, w2, w3)
            ).item()

            history.append((
                epoch,
                lr,
                w1.item(),
                w2.item(),
                w3.item(),
                current_loss
            ))

    with torch.no_grad():
        final_pred = predict(w1, w2, w3).tolist()

    final_w = (w1.item(), w2.item(), w3.item())

    return final_w, history, final_pred

learned_w, history, final_pred = train_w()

print("\n--- 训练过程 ---")
print("轮数    学习率       w1          w2          w3          损失")

for index in [0, 1, 4, 9, 99, 999]:
    epoch, lr, a, b, c, current_loss = history[index]

    print(
        f"{epoch:4d}  {lr:.6f}  "
        f"{a:10.6f}  {b:10.6f}  {c:10.6f}  "
        f"{current_loss:.6f}"
    )

print("\n最终学到的 w1：", learned_w[0])
print("最终学到的 w2：", learned_w[1])
print("最终学到的 w3：", learned_w[2])
print("最终预测值：", final_pred)
print("最终损失：", history[-1][-1])

print("\n--- 不同初始值的实验 ---")

for initial_w in [1.0, -1.0, 0.0]:
    final_w, records, _ = train_w(initial_w=initial_w)

    print(
        f"初始 w={initial_w:5.1f} -> "
        f"最终 w1={final_w[0]:.6f}, "
        f"w2={final_w[1]:.6f}, "
        f"w3={final_w[2]:.6f}, "
        f"最终损失={records[-1][-1]:.6f}"
    )


_, fixed_history, _ = train_w(
    initial_w=1.0,
    epochs=40,
    base_lr=0.1,
    decay=False
)

print("\n--- 固定学习率实验：最后 8 轮 ---")
print("轮数       w1          w2          w3          损失")

for epoch, lr, a, b, c, current_loss in fixed_history[-8:]:
    print(
        f"{epoch:4d}  {a:10.6f}  {b:10.6f}  "
        f"{c:10.6f}  {current_loss:.6f}"
    )

当前 w1： 1.0
当前 w2： 1.0
当前 w3： 1.0
预测值： [3.0, 1.75, 1.0, 0.75, 1.0, 1.75, 3.0, 4.75, 7.0]
损失： 2.0

--- 更新前 ---
w1： 1.0
w2： 1.0
w3： 1.0
损失： 2.0
w1 的梯度： -0.111111119389534
w2 的梯度： 1.4444444179534912
w3 的梯度： 0.222222238779068

--- 更新后 ---
w1： 1.0111111402511597
w2： 0.855555534362793
w3： 0.9777777791023254
预测值： [2.377777576446533, 1.3861110210418701, 0.8222221732139587, 0.6861110925674438, 0.9777777791023254, 1.6972222328186035, 2.844444513320923, 4.419444561004639, 6.42222261428833]
损失： 1.802469253540039

--- 训练过程 ---
轮数    学习率       w1          w2          w3          损失
   1  0.100000    1.011111    0.855556    0.977778  1.802469
   2  0.050000    1.022222    0.788889    0.972222  1.710494
   5  0.020000    1.039630    0.684444    0.963519  1.566399
  10  0.010000    1.053977    0.598360    0.956345  1.447634
 100  0.001000    1.104164    0.297239    0.931251  1.032198
1000  0.000100    1.183404    0.114636    0.978868  0.821520

最终学到的 w1： 1.1834039688110352
最终学到的 w2： 0.11463580280542374


### 我的观察与说明

请在这里填写你的回答。改变函数方程式，增加w变量为w1,w2,w3,

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。